# Session 15 · Case study: semantic search over customs decisions with SQLite and NumPy

**Worked example for block 1.** Build semantic search over past Binding Tariff Information decisions: chunk the descriptions, embed them with a multilingual model, store chunks, metadata and embeddings in an SQLite file, and search in any language, with and without a chapter filter. SQL selects the rows, NumPy ranks them.

**Storage.** SQLite is part of Python (`sqlite3`); nothing has to be installed or started. The SQLite file and the cached embeddings are written to `embeddings_cache/` next to this notebook, which git ignores. Needs `sentence-transformers` (model `intfloat/multilingual-e5-small`, about 470 MB on first use). Embedding the 12,360 chunks takes about 40 seconds with a GPU and several minutes on a CPU, once.

Theory: [01-lecture-retrieval-augmented-generation-and-evaluation.ipynb](01-lecture-retrieval-augmented-generation-and-evaluation.ipynb), sections 2 to 4.

*Author: course team, licence CC-BY-4.0.*

In [1]:
import sqlite3
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
CACHE = ROOT / "sessions" / "15-rag-and-agents" / "workbooks" / "embeddings_cache"   # ignored by git
CACHE.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / "sessions" / "15-rag-and-agents" / "workspace" / "src"))   # the workspace package

decisions = pd.read_parquet(DATA / "train_sample.parquet")
year = decisions["start_date"].dt.year
docs = decisions[year <= 2021].sample(10_000, random_state=0).reset_index(drop=True)      # the collection
print(len(docs), "decisions in the collection")

10000 decisions in the collection


## 1. Chunk the descriptions

Most descriptions are shorter than 120 words and stay one chunk; long German descriptions with examination results are split.

In [2]:
from bti_assistant.chunking import chunk_documents

chunks = chunk_documents(dict(zip(docs["bti_reference"], docs["description"])), size=120, overlap=20)
meta = docs.set_index("bti_reference")
chunk_df = pd.DataFrame({"chunk_id": [c.chunk_id for c in chunks], "bti_reference": [c.doc_id for c in chunks],
                         "chunk": [c.text for c in chunks]})
chunk_df = chunk_df.join(meta[["heading", "issuing_country", "language"]], on="bti_reference")
print(len(docs), "decisions ->", len(chunk_df), "chunks")
chunk_df["chunk"].str.split().str.len().describe().round(1)

10000 decisions -> 12360 chunks


count    12360.0
mean        75.2
std         33.8
min          2.0
25%         46.0
50%         75.0
75%        109.0
max        120.0
Name: chunk, dtype: float64

## 2. Embed the chunks

In [3]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)


def embed(name, texts, prefix):
    """Encode once, then load from disk; the file name records model, prefix and content."""
    path = CACHE / f"{MODEL_NAME.split('/')[-1]}_{prefix.strip(': ')}_{name}.npy"
    if path.exists():
        return np.load(path)
    E = model.encode([prefix + t for t in texts], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(path, E)
    return E


E = embed("collection_120w", chunk_df["chunk"].tolist(), "passage: ")
print(E.shape, "| vector length:", round(float(np.linalg.norm(E[0])), 3))

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11567.85it/s]

(12360, 384) | vector length: 1.0


## 3. Store chunks, metadata and embeddings in SQLite

One table holds everything a search needs; each embedding is a BLOB of 384 `float32` numbers (1,536 bytes). The file stays on disk, so a second run of the notebook (or a web service in Session 16) can search without embedding the collection again.

In [4]:
DB_PATH = CACHE / "decision_chunks.sqlite"
con = sqlite3.connect(DB_PATH)
con.execute("DROP TABLE IF EXISTS decision_chunks")
con.execute("""CREATE TABLE decision_chunks (chunk_id TEXT PRIMARY KEY, bti_reference TEXT, heading TEXT,
               issuing_country TEXT, language TEXT, chunk TEXT, embedding BLOB)""")
with con:                                                   # one transaction for all rows
    con.executemany("INSERT INTO decision_chunks VALUES (?, ?, ?, ?, ?, ?, ?)",
                    [(r.chunk_id, r.bti_reference, r.heading, r.issuing_country, r.language, r.chunk,
                      E[i].astype(np.float32).tobytes()) for i, r in enumerate(chunk_df.itertuples())])
con.execute("CREATE INDEX IF NOT EXISTS idx_chapter ON decision_chunks (substr(heading, 1, 2))")
print(con.execute("SELECT count(*), sum(length(embedding)) / 1e6 AS mb FROM decision_chunks").fetchone())
print(f"file size: {DB_PATH.stat().st_size / 1e6:.0f} MB")

(12360, 18.98496)
file size: 42 MB


## 4. Search: SQL selects, NumPy ranks

The chapter filter is a `WHERE` clause on the first two digits of the heading; the index on that expression lets SQLite find the rows of one chapter without reading the whole table. After the filter, all selected vectors are compared with the query (exact search), and only the best chunk per decision is kept.

In [5]:
def search(query, k=5, chapter=None):
    sql = "SELECT bti_reference, heading, language, chunk, embedding FROM decision_chunks"
    params = []
    if chapter:
        sql += " WHERE substr(heading, 1, 2) = ?"
        params.append(chapter)
    rows = con.execute(sql, params).fetchall()
    matrix = np.frombuffer(b"".join(r[4] for r in rows), dtype=np.float32).reshape(len(rows), -1)
    q = model.encode("query: " + query, normalize_embeddings=True)
    scores = matrix @ q                                       # cosine similarity: all vectors have length 1
    hits = pd.DataFrame([r[:4] for r in rows], columns=["bti_reference", "heading", "language", "chunk"])
    hits["score"] = scores.round(3)
    hits = hits.sort_values("score", ascending=False).drop_duplicates("bti_reference").head(k)   # one hit per decision
    hits["chunk"] = hits["chunk"].str.replace(r"\s+", " ", regex=True).str[:100]
    return hits.reset_index(drop=True)


pd.set_option("display.max_colwidth", 110)
search("children's toy car made of plastic")

,bti_reference,heading,language,chunk,score
0,FRBTIFR-BTI-2020-05759,9503,fr,"Véhicule jouet pour enfants de plus de 3 ans, se présentant sous la forme d’un tracteur à roues libr",0.861
1,XIBTI505019583,9503,en,The product is a children’s toy which helps children learn about the entire process of decomposition,0.860
2,GBBTI503598933,9503,en,A PLASTIC TOY IN THE FORM OF A BUS MEASURING ABOUT 30 X 16 X 10 CENTIMETRES. IT IS AN EDUCATIONAL EL,0.858
3,GBBTI504518469,3904,en,PLASTICISED PVC GRANULES FOR USE ON THE SIDE GLASS GASKET OF A CAR.,0.851
4,GBBTI504580171,9503,en,A TOY SET COMPRISING OF 3 KEYCHAINS THAT CHILDREN CREATE USING COLOURFUL GLITTER DOTS. THERE ARE THR,0.845


In [6]:
search("Damenstiefel mit Oberteil aus Leder", chapter="64")

,bti_reference,heading,language,chunk,score
0,DEBTI6999/19-1,6403,de,"Stiefel (lt. Antrag: ""Freizeitstiefelette für Damen"", Foto siehe Anlage) - mit Laufsohlen aus Kunsts",0.885
1,DEBTI7002/21-1,6403,de,"Stiefel, ""Damenschuhe"", Foto siehe Anlage, - mit Laufsohlen aus Kautschuk/Kunststoff, ohne Hauptsohl",0.883
2,DEBTI7004/21-1,6403,de,"Stiefel, ""Damenschuhe"", Größe 4,5, Foto siehe Anlage - mit Laufsohlen aus Kautschuk/Kunststoff; ohne",0.882
3,DEBTI4706/20-1,6403,de,"Stiefel (lt. Antrag: ""Stiefelette"", Größe 4,5, Foto siehe Anlage) - mit Laufsohlen aus Kautschuk/Kun",0.882
4,DEBTI25986/19-1,6403,de,"Stiefel (lt. Antrag: ""Reitstiefel"", Größe 38, Foto siehe Anlage) - in einem einfachen Beutel [üblich",0.882


In [7]:
search("chaussures de sport avec dessus en matière textile")    # French query

,bti_reference,heading,language,chunk,score
0,FRBTIFR-BTI-2019-17211,6405,fr,"Chaussure d’intérieur, du genre ballerine, dont le dessus et la semelle extérieure sont en matière t",0.879
1,FRBTIFR-BTI-2020-05575,6404,fr,"Chaussure de sécurité à lacets, couvrant la cheville, unisexe, pour adulte, dont le dessus est en ma",0.868
2,FRBTIFR-BTI-2020-00166,6404,fr,"Chaussure légère de type mule, comportant une bande, sur le dessus du pied, en matière textile (four",0.864
3,FRBTIFR-BTI-2019-08433,6404,fr,"Chaussure fourrée comportant une semelle extérieure en matière plastique, un dessus en matière texti",0.862
4,FRBTIFR-BTI-2021-01091,6402,fr,"Chaussure pour enfant de type tennis ne couvrant pas la cheville, avec une fermeture à lacet, dont l",0.861


**Question.** Look at the languages of the hits. Does an English query find German decisions? (Session 14 found that neighbours are usually in the language of the query.)

## 5. How fast is exact search?

Loading the rows from SQLite and ranking them in NumPy, with and without the chapter filter. Most of the time goes into reading and converting the BLOBs; a service that answers many requests keeps the matrix in memory and runs only the matrix product per request.

In [8]:
def timed(fn, n=10):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return round((time.perf_counter() - t0) / n * 1000, 1)


q = model.encode("query: toy car", normalize_embeddings=True)
print("search without filter, ms:", timed(lambda: search("toy car")))
print("search in chapter 64, ms: ", timed(lambda: search("toy car", chapter="64")))
print("matrix product only, ms:  ", timed(lambda: E @ q, n=100))
con.close()

search without filter, ms: 111.4


search in chapter 64, ms:  35.4
matrix product only, ms:   0.9
